# 04 - Search Query Expansion & Keyword Generation

Generates search queries in `"viaje a [destination]"` format from the keyword master list to feed PYTRENDS and the DataForSEO API.

### Architectural Note: reason for a separate module:
Notebook dedicated to **Query Expansion**. While Notebook 03 focused on data cleaning and baseline mapping, this step translates those raw 
destinations into actual **Google Ads bidding formats**.

This logic was isolated to keep the pipeline scalable. Currently, it generates the primary awareness intent (`"viaje a " + keyword`). With this we can easily use cross-join vectorization in the future to add multiple commercial intents (e.g., *"circuito a"*, *"viaje en grupo a"*, *"vuelos a"*, etc.) without breaking the foundational dataset.

> **Design decision:** The prefix `"viaje a "` was intentionally chosen to capture **organized / agency-mediated travel intent** for medium and long-haul destinations — the segment most relevant to tour operators and travel agencies negotiating airline seat allotments.

> **Empirical validation (INE ETR 2019–2025):** Among leisure travelers, ~45% of trips to América and ~37% to Resto del Mundo are booked through travel packages, compared to only ~18% for European destinations. > This confirms that the "viaje a" prefix aligns with the  agency-mediated segment targeted by this project, where travel package rates are 2–2.5× higher than for European destinations.

> **Known comparison asymmetry:** The INE mobility dataset captures *all* outbound tourism regardless of trip type or distance. Correlations should therefore be evaluated **against long-haul or extra-European destinations only**, where the search signal and the mobility signal refer more to the same traveler segment.

## 1. Imports

In [1]:
from pathlib import Path
from unidecode import unidecode

import pandas as pd

## 2. Paths Configuration

In [2]:
PROCESSED_PATH = Path("../data/processed")

OUTPUT_PATH = Path("../outputs/keywords")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

## 3. Data Ingestion

In [3]:
df_keywords = pd.read_parquet(
    PROCESSED_PATH / "03_keyword_mapping_master.parquet"
)

## 4. Unique Keywords Extraction
Isolating the core terminology and dropping any missing search terms to prepare for expansion.

In [4]:
df_keywords_unique = (
    df_keywords[[
        "search_term",
        "parent_country",
        "keyword_type"
    ]]
    .dropna(subset=["search_term"])
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5. Query Expansion (Instant Generation)
Instead of processing keywords one by one, the entire list of base keywords is merged with the commercial prefixes all at once. This method is fast and makes it effortless to add new prefixes whenever the marketing strategy grows.

#### 1. Define commercial prefixes (scalable to multiple intents in the future)

In [5]:
# 1. Prefix definition as a small Pandas df for future escalability
df_prefixes = pd.DataFrame({"prefix": ["viaje a "]})

# 2. Cartesian product: Match every keyword with every prefix instantly
df_google_ads = df_keywords_unique.merge(df_prefixes, how="cross")

# 3. String concatenation and normalization
df_google_ads["google_ads_keyword"] = (
    (df_google_ads["prefix"] + df_google_ads["search_term"])
    .astype(str)
    .str.lower()
    .str.strip()
)

# 4. Final cleanup and formatting
df_google_ads = (
    df_google_ads
    .drop(columns=["prefix"])
    .drop_duplicates(subset=["google_ads_keyword"])
    .sort_values(["parent_country", "google_ads_keyword"])
    .reset_index(drop=True)
)

## 6. Data Quality Review

In [6]:
print(f"Total Google Ads Keywords Generated: {df_google_ads.shape[0]}")
print(f"Total Columns: {df_google_ads.shape[1]}\n")

print("--- NULL VALUES ---")
print(df_google_ads.isnull().sum())

display(df_google_ads.head(15))

Total Google Ads Keywords Generated: 220
Total Columns: 4

--- NULL VALUES ---
search_term           0
parent_country        0
keyword_type          0
google_ads_keyword    0
dtype: int64


,search_term,parent_country,keyword_type,google_ads_keyword
0,albania,albania,country,viaje a albania
1,balcanes,albania,region,viaje a balcanes
2,alemania,alemania,country,viaje a alemania
3,andorra,andorra,country,viaje a andorra
4,angola,angola,country,viaje a angola
5,arabia saudi,arabia saudi,country,viaje a arabia saudi
6,arabia saudita,arabia saudi,country,viaje a arabia saudita
7,argelia,argelia,country,viaje a argelia
8,argentina,argentina,country,viaje a argentina
9,crucero australis,argentina,comercial,viaje a crucero australis


## 7. Export

In [7]:
df_export = df_google_ads[[
    "google_ads_keyword"
]].rename(columns={
    "google_ads_keyword": "Keyword" # DataForSEO API expects this name
})

df_export.to_csv(
    OUTPUT_PATH / "04_keywords_with_prefix.csv", # DataForSEO API input
    index=False,
    encoding="utf-8-sig"
)

df_google_ads.to_parquet(
    OUTPUT_PATH / "04_keywords_with_prefix.parquet"
)

df_google_ads.to_excel(
    OUTPUT_PATH / "04_keywords_with_prefix.xlsx", # copy for manual checking
    index=False
)

print(f"✔ Keywords successfully exported to: {OUTPUT_PATH}")

✔ Keywords successfully exported to: ..\outputs\keywords


## Next Steps
Use the generated `04_keywords_with_prefix.csv` file as the primary payload to execute Search Volume extraction via the **DataForSEO Google Ads API**.